# QwenVLA S0 on LIBERO+
Select an A100 runtime and grant this notebook access to the existing Colab Secret HF_TOKEN. The default run stops at optimizer step 32 and saves a resumable checkpoint. To continue, set STOP_AFTER to None and RESUME_STEP to 32, then rerun the training cell. For a new runtime, restore the sealed checkpoint from the private Hub and reuse the recorded Git SHA.


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

SOURCE_REF = "dev/fish13031e"
ROOT = Path("/content/vlade_qwen_vla_source")
REPOSITORY = "https://github.com/eastha10/VLADE.git"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["VLADE_ASSET_CACHE"] = "/content/vlade_hf_assets"
os.environ["VLADE_HUB_REPO"] = "VLADE-konkuk/VLADE-checkpoints"
os.environ["VLADE_HUB_RUN_ID"] = "qwen-vla-s0-seed42-r1"

def checked(*args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.free", "--format=csv,noheader,nounits"], text=True
).splitlines()[0]
name, free = gpu.rsplit(",", 1)
if "A100" not in name or int(free) < 22000:
    raise RuntimeError(f"Select an A100 with at least 22,000 MiB free. Current GPU: {gpu}")

if not ROOT.exists():
    checked("git", "clone", "--no-checkout", "--single-branch", "--branch", "dev/fish13031e", REPOSITORY, str(ROOT))
    ref = "origin/dev/fish13031e" if SOURCE_REF == "dev/fish13031e" else SOURCE_REF
    if ref != "origin/dev/fish13031e" and not re.fullmatch(r"[0-9a-f]{40}", ref):
        raise ValueError("Use dev/fish13031e or the exact recorded commit")
    checked("git", "merge-base", "--is-ancestor", ref, "origin/dev/fish13031e", cwd=ROOT)
    checked("git", "checkout", "--detach", ref, cwd=ROOT)

commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
if SOURCE_REF != "dev/fish13031e" and commit != SOURCE_REF:
    raise RuntimeError(f"Existing checkout is pinned to {commit}")
os.environ["PYTHONPATH"] = str(ROOT / "src")
print("SOURCE_COMMIT", commit, flush=True)
checked(sys.executable, "-m", "pip", "install", "torch==2.12.0", "torchvision==0.27.0", "transformers==5.9.0")


In [ ]:
STOP_AFTER = 32
RESUME_STEP = None
output = Path("/content/vlade_runs/qwen-vla-s0-seed42")
args = [sys.executable, "-m", "vlade.training.qwen_vla_task",
        "--config", "configs/experiments/qwen_vla_s0.json", "--output", str(output)]
if STOP_AFTER is not None:
    args += ["--stop-after", str(STOP_AFTER)]
if RESUME_STEP is not None:
    args += ["--resume", str(output / f"step-{RESUME_STEP:07d}")]
checked(*args, cwd=ROOT)


In [ ]:
import json

for log in sorted(output.glob("metrics-*.jsonl")):
    rows = {row["step"]: row for row in map(json.loads, log.read_text().splitlines())}
    if {10, 30} <= rows.keys():
        seconds_per_step = (rows[30]["elapsed_seconds"] - rows[10]["elapsed_seconds"]) / 20
        print({"seconds_per_optimizer_step": seconds_per_step,
               "projected_8000_training_hours": seconds_per_step * 8000 / 3600,
               "peak_allocated_gib": rows[max(rows)]["peak_allocated_bytes"] / 1024**3})
